# IMRPhenomXPHM / SpinTaylor fitting factor against NRSur7dq4

This is a workflow template for expensive model comparisons. It keeps waveform generation behind user-supplied hooks, then shows how to run three fitting-factor searches:

1. optimize all intrinsic candidate parameters,
2. optimize intrinsic parameters plus full 3D Wigner rotations,
3. optimize only spin angles and phase-like parameters for a fixed intrinsic configuration.

The cells are intentionally output-free in the repository. Fill in the model-generation hooks for your local LALSuite/surrogate setup before running the searches.

In [ ]:
from __future__ import annotations

import importlib
from pprint import pprint

import numpy as np

import waveformtools.comparison  # installs ModesArray.fitting_factor
from waveformtools.comparison import AlignmentSpec, FittingFactorConfig, ModeComparisonConfig, RotationSpec


def optional_import(name):
    try:
        return importlib.import_module(name)
    except ImportError as exc:
        raise RuntimeError(f"Install or activate the dependency needed for {name!r} before running this notebook.") from exc


# Uncomment when running real waveform generation.
# lal = optional_import("lal")
# lalsimulation = optional_import("lalsimulation")
# gwsurrogate = optional_import("gwsurrogate")


## User-supplied waveform hooks

Both hooks should return `waveformtools.modes_array.ModesArray` objects on compatible time axes and with the modes you want to compare. Keep all model-specific conversion code here.

In [ ]:
def load_nrsur7dq4_reference(**nr_parameters):
    """Return the target NRSur7dq4 ModesArray for the requested parameters."""
    raise NotImplementedError("Load or generate NRSur7dq4 modes here, then return a ModesArray.")


def generate_phenomxphm_spintaylor_candidate(**candidate_parameters):
    """Return an IMRPhenomXPHM/SpinTaylor candidate ModesArray."""
    raise NotImplementedError("Generate IMRPhenomXPHM/SpinTaylor modes here, then return a ModesArray.")


def print_fitting_result(label, result):
    print(f"\n{label}")
    print("match:", result.match)
    print("mismatch:", result.mismatch)
    print("optimizer:", result.optimizer)
    print("waveform generations:", result.n_waveform_generations)
    print("best candidate/generator parameters:")
    pprint(result.best_parameters.get("generator", {}))
    print("best alignment/rotation parameters:")
    pprint(result.best_parameters.get("alignment", {}))


## Reference and baseline settings

In [ ]:
nr_reference_parameters = {
    "q": 2.0,
    "chi1x": 0.1,
    "chi1y": 0.0,
    "chi1z": 0.3,
    "chi2x": -0.05,
    "chi2y": 0.02,
    "chi2z": -0.2,
}

# target_modes = load_nrsur7dq4_reference(**nr_reference_parameters)

comparison_base = ModeComparisonConfig(
    ell_min=2,
    ell_max=4,
    alignment=AlignmentSpec(
        time_alignment="peak_total_news_power",
        time_domain_policy="resample_to_reference",
        phase_alignment="orbital_phase_and_global",
        resample_method="linear",
        optimize_time_shift=True,
    ),
)


## Search 1: optimize all intrinsic candidate parameters

In [ ]:
all_intrinsic_config = FittingFactorConfig(
    comparison=comparison_base,
    variable_parameters={
        "q": (1.0, 6.0),
        "chi1x": (-0.8, 0.8),
        "chi1y": (-0.8, 0.8),
        "chi1z": (-0.8, 0.8),
        "chi2x": (-0.8, 0.8),
        "chi2y": (-0.8, 0.8),
        "chi2z": (-0.8, 0.8),
    },
    initial_parameters=nr_reference_parameters,
    fixed_parameters={"approximant": "IMRPhenomXPHM", "spin_evolution": "SpinTaylor"},
    optimizer="differential_evolution",
    optimizer_options={"maxiter": 80, "polish": True, "workers": 1},
)

# result_all_intrinsic = target_modes.fitting_factor(
#     generate_phenomxphm_spintaylor_candidate,
#     config=all_intrinsic_config,
# )
# print_fitting_result("All intrinsic parameters", result_all_intrinsic)


## Search 2: optimize intrinsic parameters plus full 3D Wigner rotation

In [ ]:
intrinsic_plus_rotation_config = FittingFactorConfig(
    comparison=ModeComparisonConfig(
        ell_min=2,
        ell_max=4,
        alignment=comparison_base.alignment,
        rotation=RotationSpec(
            kind="wigner",
            optimize_parameters=("alpha", "beta", "gamma"),
            parameter_bounds={
                "alpha": (-np.pi, np.pi),
                "beta": (-np.pi, np.pi),
                "gamma": (-np.pi, np.pi),
            },
        ),
    ),
    variable_parameters=all_intrinsic_config.variable_parameters,
    initial_parameters=all_intrinsic_config.initial_parameters,
    fixed_parameters=all_intrinsic_config.fixed_parameters,
    optimizer="differential_evolution",
    optimizer_options={"maxiter": 80, "polish": True, "workers": 1},
)

# result_intrinsic_plus_rotation = target_modes.fitting_factor(
#     generate_phenomxphm_spintaylor_candidate,
#     config=intrinsic_plus_rotation_config,
# )
# print_fitting_result("Intrinsic plus full 3D Wigner rotation", result_intrinsic_plus_rotation)


## Search 3: fixed intrinsic parameters, optimize spin angles and phase-like parameters

In [ ]:
fixed_intrinsic_parameters = {
    "q": nr_reference_parameters["q"],
    "chi1_magnitude": 0.35,
    "chi2_magnitude": 0.25,
    "approximant": "IMRPhenomXPHM",
    "spin_evolution": "SpinTaylor",
}

spin_angles_and_phase_config = FittingFactorConfig(
    comparison=ModeComparisonConfig(
        ell_min=2,
        ell_max=4,
        alignment=comparison_base.alignment,
        rotation=RotationSpec(kind="z_axis", optimize_angle=True, angle_bounds=(-np.pi, np.pi)),
    ),
    variable_parameters={
        "spin1_theta": (0.0, np.pi),
        "spin1_phi": (-np.pi, np.pi),
        "spin2_theta": (0.0, np.pi),
        "spin2_phi": (-np.pi, np.pi),
        "reference_phase": (-np.pi, np.pi),
    },
    initial_parameters={
        "spin1_theta": 0.7,
        "spin1_phi": 0.0,
        "spin2_theta": 1.0,
        "spin2_phi": 0.0,
        "reference_phase": 0.0,
    },
    fixed_parameters=fixed_intrinsic_parameters,
    optimizer="differential_evolution",
    optimizer_options={"maxiter": 80, "polish": True, "workers": 1},
)

# result_spin_angles_phase = target_modes.fitting_factor(
#     generate_phenomxphm_spintaylor_candidate,
#     config=spin_angles_and_phase_config,
# )
# print_fitting_result("Spin angles and phases", result_spin_angles_phase)
